# `epibyhand`: the complete tutorial

**Classical epidemiological measures that show their work.**

Most statistical software gives you the answer. In a methods course the answer
is the least interesting part of the calculation — a student who can produce
`2.14` without being able to say where it came from has learned nothing that
will survive the exam.

Every function in `epibyhand` returns the answer *together with the reasoning
that produced it*: each intermediate quantity, the formula in symbols, and the
formula with the observed numbers substituted in.

This tutorial covers **every exported function in the package**.

| Part | Topic |
|---|---|
| 1 | Building a 2 x 2 table — `epi2x2()` |
| 2 | Risk ratio — `risk_ratio()` |
| 3 | Risk difference — `risk_difference()` |
| 4 | Odds ratio — `odds_ratio()` |
| 5 | Confidence levels, and one trap |
| 6 | Zero cells and continuity corrections |
| 7 | Controlling output — `verbose`, `digits` |
| 8 | Checking hand calculations — `check_work()` |
| 9 | Attributable fractions — `attributable_fraction()` |
| 10 | Stratified data — `epi_strata()`, `collapse_strata()` |
| 11 | Pooling odds ratios — `mh_odds_ratio()` |
| 12 | Pooling risk ratios — `mh_risk_ratio()` |
| 13 | Homogeneity — `homogeneity()` |
| 14 | Programmatic use — `estimate()`, `confint()`, `steps_table()` |
| 15 | Extending the package — `derivation()`, `derivation_step()` |
| 16 | A complete analysis, start to finish |

Followed by a **function reference** and **ten exercises with worked answers**.

### Scope

`epibyhand` covers **methods a student can compute by hand on paper**. That
boundary is deliberate: it is why there is no regression modelling here. Once
an estimate comes from an iterative fit there is no hand calculation left to
check, and a printed "derivation" would be decoration rather than instruction.

The package imports only `stats`. Nothing else.

---
## Setup

**This notebook needs an R runtime.** In Colab go to
**Runtime → Change runtime type → Runtime type: R**, then run the cell below.

(If the menu has no R option, open a fresh R notebook from
`https://colab.research.google.com/#create=true&language=r` and paste this
notebook's cells into it.)

In [ ]:
# Install from CRAN. Takes about 30 seconds; there is nothing to compile.
if (!requireNamespace("epibyhand", quietly = TRUE)) {
  install.packages("epibyhand")
}

library(epibyhand)
packageVersion("epibyhand")

In [ ]:
# Everything the package exports
sort(getNamespaceExports("epibyhand"))

Fifteen functions. This tutorial uses all of them.

---
## Part 1 — The 2 x 2 table

Everything starts here. `epi2x2()` takes the four cell counts in the standard
epidemiological orientation used by Rothman and Greenland:

|  | Case | Non-case |
|---|---|---|
| **Exposed** | a | b |
| **Unexposed** | c | d |

So the argument order is **a, b, c, d** — across the top row, then across the
bottom row.

### Our first dataset

A church picnic is followed by an outbreak of gastroenteritis. Investigators
interview all 120 attendees and ask whether they ate the potato salad:

* Of the 70 who **ate** the potato salad, **54 became ill**.
* Of the 50 who **did not**, **8 became ill**.

In [ ]:
picnic <- epi2x2(54, 16, 8, 42,
                 exposure = c("Ate potato salad", "Did not eat"),
                 outcome  = c("Ill", "Well"))
picnic

The `exposure` and `outcome` arguments only change the labels — they do not
change any arithmetic. But labelling the table honestly is worth the extra few
characters, because it is what stops you from reading the output of a
case-control study as though it were a cohort.

### Input from a matrix

If your data already live in a matrix or a `table`, pass it directly. Note
`byrow = TRUE` — the natural reading order matches a, b, c, d.

In [ ]:
m <- matrix(c(54, 16,
              8, 42), nrow = 2, byrow = TRUE)
epi2x2(m)

If the matrix carries dimnames, `epi2x2()` picks up the labels automatically,
so you do not have to repeat yourself:

In [ ]:
m2 <- matrix(c(54, 16, 8, 42), nrow = 2, byrow = TRUE,
             dimnames = list(c("Ate potato salad", "Did not eat"),
                             c("Ill", "Well")))
epi2x2(m2)

### What the object actually is

No S4, no reference classes — a plain list with a class attribute. You can
reach into it whenever that is more convenient than a function call.

In [ ]:
str(unclass(picnic))
picnic$a
picnic$a + picnic$b     # total exposed

### Input validation

The constructor refuses malformed input rather than producing a wrong answer
quietly:

In [ ]:
try(epi2x2(54, 16, 8))                      # only three counts
try(epi2x2(matrix(1:6, nrow = 2)))          # not 2 x 2
try(epi2x2(-5, 16, 8, 42))                  # negative count

**A note on orientation.** Getting a, b, c, d in the wrong order is the single
most common source of a wrong answer in this whole tutorial. If your odds ratio
comes out as the reciprocal of what you expected, you have almost certainly
swapped the rows or the columns. Nothing in the software can catch this for
you — the table is valid either way.

---
## Part 2 — Risk ratio

This is a **cohort**: we followed a defined group forward and counted who fell
ill. Risk is therefore estimable — cases divided by people at risk — and the
risk ratio is available to us.

In [ ]:
risk_ratio(picnic)

Read the derivation top to bottom and notice what it is doing.

**Steps 1 and 2** compute the two risks separately before dividing them. This
matters pedagogically: a student who computes `54/70 = 0.7714` and stops has
done step 1 correctly. Telling them "wrong, the answer is 4.82" hides that from
both of you.

**Step 3** is the ratio.

**Steps 4-6** build the confidence interval on the **log scale** and then
exponentiate. That is why the interval is not symmetric around the estimate:
4.82 sits closer to 2.52 than to 9.22 arithmetically, but exactly in the middle
on the log scale. Ratios are multiplicative, and their sampling distribution is
far closer to normal after a log transform.

---
## Part 3 — Risk difference

The same table, asked as a difference rather than a ratio.

In [ ]:
risk_difference(picnic)

The risk difference is **0.61** — 61 additional cases per 100 people exposed.

The output also reports the **number needed to expose**, the reciprocal of the
risk difference. Here roughly 2 people had to eat the potato salad to produce
one extra case. (In a treatment context the same arithmetic is the number
needed to treat.)

### Ratios and differences answer different questions

* **Risk ratio (4.82)** — how many *times* more likely are the exposed to fall
  ill? A question about strength of association, relevant to causation.
* **Risk difference (0.61)** — how many *extra cases* does the exposure
  produce? A question about impact, relevant to deciding where to spend money.

A rare exposure can have a huge risk ratio and a negligible risk difference.
Neither number is more correct; the confusion between them is behind a great
deal of bad science communication.

Unlike the ratio measures, the risk difference interval is symmetric — it is
built on the natural scale, because a difference is already additive.

---
## Part 4 — Odds ratio

Odds are not risks. The **odds** of an event are the number of times it happens
divided by the number of times it does not — `a/b`, not `a/(a+b)`.

In [ ]:
odds_ratio(picnic)

The odds ratio is **17.72**, while the risk ratio for the very same table was
**4.82**.

That is not an error. **The odds ratio is always further from 1 than the risk
ratio**, and the gap widens as the outcome becomes more common. Here 77% of the
exposed fell ill — an extremely common outcome — so the two diverge
dramatically.

Notice the package said so itself, unprompted, in the note beneath the result.
Because the data are tabulated as a cohort it knows risk is estimable, so it
reports what the risk ratio would be rather than letting you walk off with the
odds ratio unexamined.

### Why use the odds ratio at all?

Because it has a property no other measure has: it is **estimable from a
case-control study**. When you sample on the outcome — deliberately recruiting
cases and controls in whatever ratio is convenient — you destroy your ability
to estimate risk. But the odds ratio is unchanged whether you condition on
exposure or on outcome, which is what the note in step 3 means by `OR = ad/bc`.

### The rare disease assumption, demonstrated

Fix the risk ratio at exactly 2.0 and vary only how common the outcome is:

In [ ]:
options(epibyhand.verbose = 0)   # results only, no derivations

for (R0 in c(0.001, 0.01, 0.05, 0.10, 0.20, 0.40)) {
  n <- 100000
  c_cell <- R0 * n;      d_cell <- n - c_cell
  a_cell <- 2 * R0 * n;  b_cell <- n - a_cell
  tab <- epi2x2(a_cell, b_cell, c_cell, d_cell)
  cat(sprintf("baseline risk %5.3f   RR = %.3f   OR = %.4f\n",
              R0, estimate(risk_ratio(tab)), estimate(odds_ratio(tab))))
}

options(epibyhand.verbose = 2)

The risk ratio is pinned at 2.000 in every row. The odds ratio drifts from
**2.002** to **6.000**.

This is the rare disease assumption made concrete. When the outcome is rare
(under roughly 5-10%), the odds ratio approximates the risk ratio, which is why
case-control studies of rare diseases can report an OR and have
epidemiologists read it as a risk ratio. When the outcome is common, treating
an odds ratio as though it were a risk ratio **badly overstates the effect** —
and this happens constantly in published abstracts.

---
## Part 5 — Confidence levels, and one trap

Every measure takes a `conf_level` argument:

In [ ]:
options(epibyhand.verbose = 0)

estimate(odds_ratio(picnic))
confint(odds_ratio(picnic))                      # default 95%
confint(odds_ratio(picnic, conf_level = 0.99))   # 99%
confint(odds_ratio(picnic, conf_level = 0.90))   # 90%

### The trap

`confint()` on an `epibyhand` object **ignores its `level` argument**. The
level is fixed when the derivation is built, not when you extract from it:

In [ ]:
d <- odds_ratio(picnic)                # built at 95%

confint(d, level = 0.99)               # LOOKS like 99% -- it is not
confint(d)                             # identical
confint(odds_ratio(picnic, conf_level = 0.99))   # this is the 99% interval

The first two lines are the same numbers. `confint(d, level = 0.99)` returns
the **95%** interval, because the interval was already computed and stored when
`odds_ratio()` ran.

This is a consequence of the package's design — the whole point is that the
derivation shows the arithmetic that produced *that* interval, so an extractor
cannot silently produce a different one. But it is a real trap, because
`confint()` on a model object in base R does honour `level`.

**Rule: set the level where the measure is computed, never where it is
extracted.**

---
## Part 6 — Zero cells

A zero cell makes a ratio and its standard error undefined. Many packages
silently add 0.5 to every cell and hand you a number. `epibyhand` does not.

In [ ]:
options(epibyhand.verbose = 0)
odds_ratio(epi2x2(10, 0, 5, 20))

You get `Inf`, an `NA` limit, and an explanation of what a continuity
correction would do and why you might not want it.

This is deliberate. The Haldane-Anscombe correction — adding 0.5 to every
cell — is a defensible choice, but it **biases the estimate toward the null**
and makes the confidence interval approximate. That is a decision the analyst
should make and report, not something software should do behind their back.

If you want the correction, apply it explicitly so that it appears in your code
and therefore in your methods section:

In [ ]:
corrected <- epi2x2(10 + 0.5, 0 + 0.5, 5 + 0.5, 20 + 0.5)
odds_ratio(corrected)

Note the estimate is now finite (**26.6**) with a usable interval — but it is a
number you chose to create. The uncorrected answer was `Inf`, and `Inf` is the
honest description of a table where nobody in the exposed group avoided the
outcome.

---
## Part 7 — Controlling the output

Two global options control everything the print method does.

In [ ]:
options(epibyhand.verbose = 0)   # the answer alone
odds_ratio(picnic)

In [ ]:
options(epibyhand.verbose = 1)   # add the symbolic formulas, drop the numbers
odds_ratio(picnic)

In [ ]:
options(epibyhand.verbose = 2)   # full worked solution (the default)
options(epibyhand.digits  = 3)   # and round to 3 significant digits
odds_ratio(picnic)

You can also override per call, without touching the global setting — useful
inside a script or a report where the default should stay put:

In [ ]:
options(epibyhand.digits = 4)                     # restore the default
print(odds_ratio(picnic), verbose = 1, digits = 2)

A natural teaching pattern: `verbose = 2` when introducing a measure for the
first time, `verbose = 1` when the class already knows the formula and you are
reminding them, `verbose = 0` once you are just doing analysis.

---
## Part 8 — `check_work()`

This is the feature no other package has. Give it a value and it compares
against the final answer. When that does not match, it searches **every
intermediate step** for one that does.

In [ ]:
options(epibyhand.verbose = 2)
d <- odds_ratio(picnic)

check_work(d, 17.72)

In [ ]:
check_work(d, 3.375)

That second student did not fail. They computed the odds among the exposed
(`54/16 = 3.375`) and stopped — a completely different problem from an
arithmetic slip, needing a different sentence from the person teaching them.

Compare a genuine arithmetic error:

In [ ]:
check_work(d, 9.99)

No intermediate step matches, so the tool says so and tells you to compare the
derivation line by line.

### Targeting a specific step

You can check one step by its **symbol**:

In [ ]:
check_work(d, 0.1905, step = "odds0")    # odds among the unexposed: 8/42

…or by its **number**:

In [ ]:
check_work(d, 3.375, step = 1)

Ask for a step that does not exist and it tells you what is available:

In [ ]:
try(check_work(d, 1.0, step = "RR"))

### Tolerance

The default tolerance is relative and loose enough to accept a value rounded to
two decimal places. Tighten it when you want to insist on precision:

In [ ]:
check_work(d, 17.7, tol = 0.005)     # accepted at the default tolerance
check_work(d, 17.7, tol = 0.0001)    # rejected when you demand more precision

### The return value

`check_work()` prints for humans but also returns `TRUE`/`FALSE` invisibly, so
you can build a grading script on top of it:

In [ ]:
result <- check_work(d, 17.72)
result

submissions <- c(17.72, 3.375, 9.99, 17.7188)
vapply(submissions, function(v) {
  invisible(capture.output(ok <- check_work(d, v)))
  ok
}, logical(1))

---
## Part 9 — Attributable fractions

An attributable fraction asks: **what proportion of disease would disappear if
we removed the exposure?** There are two versions and confusing them is a
classic exam mistake.

In [ ]:
attributable_fraction(picnic, among = "exposed")

**AFe = 0.79.** Among people who ate the potato salad, 79% of their illness is
attributable to it. The remaining 21% is background — they would have been ill
anyway.

Note the second step: the same quantity comes out of `(RR - 1)/RR` using only
the risk ratio. That is why AFe can be computed from a case-control study,
where absolute risks are unavailable but the ratio is estimable.

Now the population version:

In [ ]:
attributable_fraction(picnic, among = "population")

**PAF = 0.69**, lower than the AFe of 0.79.

Why? Only 58% of attendees ate the potato salad. The unexposed 42% contribute
cases to the denominator but have nothing attributable to remove, which dilutes
the fraction.

### The three formulas

The derivation computes the PAF **three ways** — directly from risks, by
Levin's formula from exposure prevalence, and by Miettinen's formula from the
proportion of cases exposed — and they agree to the last digit.

Textbooks present these as alternatives to choose between. They are not. They
are **one quantity written three ways**, and which you use depends only on
which inputs you happen to have:

* **Direct** — you have the full table.
* **Levin** — you have a risk ratio from one study and exposure prevalence from
  another.
* **Miettinen** — you have a case-control study and know what fraction of cases
  were exposed.

### Protective exposures give negative fractions

In [ ]:
options(epibyhand.verbose = 0)
protective <- epi2x2(10, 40, 30, 20)     # exposure looks protective

estimate(risk_ratio(protective))
estimate(attributable_fraction(protective, among = "exposed"))

A negative attributable fraction is not an error. The package does **not**
silently switch to reporting a *prevented* fraction, which is a different
quantity with a different formula — that kind of quiet reinterpretation is
exactly what this package exists to avoid. If you want the prevented fraction,
compute it deliberately: `PFe = 1 - RR`.

### The warning that matters

PAF depends on how common the exposure is, so **it does not transfer between
populations** the way a risk ratio does. A strong risk factor that is rare has
a small PAF; a weak one that is universal can have a large one. Quoting a PAF
from one country's study as though it applied to another is a real and common
error.

---
## Part 10 — Stratified data

### The Whickham data

In 1972-74 a survey in Whickham, England recorded whether each participant
smoked. Twenty years later the survivors were identified. What follows is the
1314 women in that cohort — the standard illustration of Simpson's paradox
(Appleton, French and Vanderpump, 1996, *The American Statistician* **50**,
340-341).

Start where a student would: smoking and death, ignoring everything else.

In [ ]:
whickham_crude <- epi2x2(139, 443, 230, 502,
                         exposure = c("Smoker", "Non-smoker"),
                         outcome  = c("Dead", "Alive"))

options(epibyhand.verbose = 1)
odds_ratio(whickham_crude)
risk_ratio(whickham_crude)

**The odds ratio is 0.68 and the confidence interval excludes 1.**

Read naively: smoking is protective, and significantly so. The risk ratio
agrees. A report written at this point would be internally consistent,
statistically significant, and **false**.

### Building stratified data

`epi_strata()` accepts four different input shapes. All produce the same
object — use whichever matches how your data already look.

In [ ]:
# 1. Four counts per stratum, as separate arguments
whickham <- epi_strata(
  c(15, 270,  12, 327),   # 18-44:  smoker dead/alive, non-smoker dead/alive
  c(80, 167,  53, 147),   # 45-64
  c(44,   6, 165,  28),   # 65+
  labels   = c("18-44", "45-64", "65+"),
  exposure = c("Smoker", "Non-smoker"),
  outcome  = c("Dead", "Alive")
)

whickham

In [ ]:
# 2. A named list -- names become the stratum labels
from_list <- epi_strata(list(
  "18-44" = c(15, 270,  12, 327),
  "45-64" = c(80, 167,  53, 147),
  "65+"   = c(44,   6, 165,  28)
))

# 3. A 2 x 2 x K array, as produced by table()
arr <- array(c(15, 12, 270, 327,
               80, 53, 167, 147,
               44, 165,  6,  28),
             dim = c(2, 2, 3),
             dimnames = list(c("Smoker", "Non-smoker"),
                             c("Dead", "Alive"),
                             c("18-44", "45-64", "65+")))
from_array <- epi_strata(arr)

# 4. A list of epi2x2 objects you built earlier
from_objects <- epi_strata(list(
  epi2x2(15, 270,  12, 327),
  epi2x2(80, 167,  53, 147),
  epi2x2(44,   6, 165,  28)
))

# All identical
c(estimate(mh_odds_ratio(whickham)),
  estimate(mh_odds_ratio(from_list)),
  estimate(mh_odds_ratio(from_array)),
  estimate(mh_odds_ratio(from_objects)))

A stratified analysis needs at least two strata, and the constructor enforces
it:

In [ ]:
try(epi_strata(c(15, 270, 12, 327)))

### Collapsing back down

`collapse_strata()` adds the strata cell by cell, discarding the stratifying
variable. The result is the **crude** table — the one you would have had if you
had never stratified.

In [ ]:
collapse_strata(whickham)

# which is exactly the crude table we started with
whickham_crude

That round trip is worth doing once in a class. It makes concrete that the
crude and stratified analyses use *the same data* — nothing was added or
removed. All that changed is whether age was allowed to be invisible.

---
## Part 11 — Mantel-Haenszel odds ratio

Look inside each age group first, before pooling anything:

In [ ]:
options(epibyhand.verbose = 0)
round(mh_odds_ratio(whickham)$stratum_estimates, 3)

**All three age groups give an odds ratio above 1.** The crude estimate was
0.68. Adjustment here does not merely shift the estimate — it **reverses** it.

Now pool them:

In [ ]:
options(epibyhand.verbose = 2)
mh_odds_ratio(whickham)

### The weights are the point

Almost no software shows you this. `S_i` is what each stratum contributes, and
the pooled estimate is a **weighted average of the stratum odds ratios** with
weights `S_i`.

That has a consequence you can check by eye: **`OR_MH` must fall between the
smallest and largest stratum estimate.** Here 1.350 sits between 1.244 and
1.514. If yours does not, your arithmetic is wrong.

The crude estimate of 0.68 does not fall in that range, and could not, because
it is **not an average of these numbers at all**. It is a different quantity
that happens to be computed from the same table.

You can verify the weighted-average identity directly:

In [ ]:
d <- mh_odds_ratio(whickham)

S_i  <- d$steps[[2]]$table$S_i      # the weights, from step 2
OR_i <- d$stratum_estimates

sum(S_i * OR_i) / sum(S_i)          # the weighted average
estimate(d)                         # what the package reports

### The confidence interval

The interval uses the **Robins-Breslow-Greenland** variance, which is
consistent both when you have a few large strata and when you have many small
ones. That generality is why it is the standard choice, and why the formula in
step 4 is so unwieldy.

The package's value matches `stats::mantelhaen.test` to ten decimal places, if
you want to confirm it against an independent implementation:

In [ ]:
arr2 <- array(c(15, 12, 270, 327,
                80, 53, 167, 147,
                44, 165,  6,  28), dim = c(2, 2, 3))
ref <- mantelhaen.test(arr2, correct = FALSE)

c(epibyhand = estimate(d), stats = unname(ref$estimate))
rbind(epibyhand = confint(d), stats = as.numeric(ref$conf.int))

### The adjusted interval crosses 1

`OR_MH = 1.350, 95% CI 0.961 to 1.896`. The crude interval excluded 1; the
adjusted one does not.

This is worth sitting with. Adjustment is about getting the **right** answer,
not a bigger or more significant one. The honest conclusion from these 1314
women is that smoking is associated with higher 20-year mortality, with an
effect estimate compatible with anything from a trivial protective effect to
nearly a doubling. The confidently significant protective effect was an
artifact.

---
## Part 12 — Mantel-Haenszel risk ratio

The same pooling logic applied to risks. This is a cohort, so risks are
estimable and the risk ratio is the more interpretable measure.

In [ ]:
mh_risk_ratio(whickham)

`RR_MH = 1.148, 95% CI 0.983 to 1.341`, using the **Greenland-Robins** variance
— the risk-ratio counterpart to Robins-Breslow-Greenland.

Compare the two pooled estimates:

| | Estimate | 95% CI |
|---|---|---|
| `mh_odds_ratio()` | 1.350 | 0.961 to 1.896 |
| `mh_risk_ratio()` | 1.148 | 0.983 to 1.341 |

The odds ratio is further from 1, exactly as in Part 4 and for the same reason:
death within 20 years is not a rare outcome in this cohort, particularly in the
oldest stratum. **For these data the risk ratio is the number you should
report.**

Both objects carry the same structure, so everything you learned above works on
either:

In [ ]:
options(epibyhand.verbose = 0)
r <- mh_risk_ratio(whickham)

round(r$stratum_estimates, 3)
r$crude
estimate(r)

---
## Part 13 — Homogeneity

A single pooled estimate only means something if **one odds ratio underlies
every stratum**. If the strata genuinely differ, the stratifying variable is an
**effect modifier**, and pooling destroys the finding rather than reporting it.

The Breslow-Day test asks whether the observed spread is more than chance.

In [ ]:
options(epibyhand.verbose = 2)
homogeneity(whickham)

X-squared = 0.118 on 2 degrees of freedom, p = 0.94. The stratum estimates
(1.51, 1.33, 1.24) are about as homogeneous as random variation allows, and no
single stratum strains against the others. Pooling is comfortable here.

Step 1 is worth reading carefully: `A_i` is what cell *a* would be if that
stratum had exactly the pooled odds ratio, holding its margins fixed. Solving
the quadratic for it is the one step in this package you would not do by hand —
everything else is arithmetic.

### Tarone's correction

Applied by default. Without it the statistic is slightly too large:

In [ ]:
options(epibyhand.verbose = 0)
c(with_tarone    = estimate(homogeneity(whickham, tarone = TRUE)),
  without_tarone = estimate(homogeneity(whickham, tarone = FALSE)))

The correction subtracts a term that accounts for the pooled estimate having
been estimated from the same data. It is small here, but it is the version
Breslow and Day's own later work recommends, so it is the default.

### The caveat the function prints anyway

**A large p-value is not evidence that the odds ratios are equal.** This test
has poor power, especially with small strata, so it will often fail to reject
whether or not effect modification is present. The stratum-specific estimates
you inspected before pooling remain the more informative thing.

### What effect modification actually looks like

Contrast a fabricated study where an exposure is harmful in younger people and
null in older ones:

In [ ]:
em <- epi_strata(
  c(60, 40, 30, 70),      # under 50:     OR = (60*70)/(40*30) = 3.5
  c(50, 50, 50, 50),      # 50 and over:  OR = (50*50)/(50*50) = 1.0
  labels = c("Under 50", "50 and over")
)

round(mh_odds_ratio(em)$stratum_estimates, 3)
estimate(mh_odds_ratio(em))

options(epibyhand.verbose = 1)
homogeneity(em)

X-squared = 9.34, p = 0.002. The strata disagree.

The Mantel-Haenszel estimate for these data is **1.81** — a number that
describes neither group. It is the average of a real effect and no effect, and
reporting it alone would hide the actual finding: **the exposure matters for
younger people and not for older ones**.

**Confounding and effect modification are different things and call for
different responses:**

| | What it is | What to do |
|---|---|---|
| **Confounding** | A nuisance distorting the crude estimate | Adjust it away, report the pooled estimate |
| **Effect modification** | A real feature of how the world works | Report the strata separately — it *is* the finding |

---
## Part 14 — Using derivations programmatically

Every function returns the same kind of object, so the same extractors work
everywhere.

In [ ]:
options(epibyhand.verbose = 0)
d <- mh_odds_ratio(whickham)

estimate(d)      # the point estimate
confint(d)       # the interval
class(d)

### `steps_table()`

Returns the whole derivation as a data frame — the basis for answer keys,
grading scripts, and rendering the working somewhere the package does not
reach.

In [ ]:
tt <- steps_table(d)
str(tt)

In [ ]:
tt[, c("step", "symbol", "label", "result")]

Six columns: `step`, `symbol`, `label`, `formula`, `substituted`, `result`.
Steps that only carry a per-stratum table (like the weights) have `NA` in
`result`, because they do not evaluate to a single number.

Here is the substituted arithmetic — the middle line of each printed step:

In [ ]:
tt[!is.na(tt$substituted), c("symbol", "substituted", "result")]

### What else the object carries

Stratified derivations attach two extras that are not in the steps table:

In [ ]:
d$stratum_estimates    # named vector, one per stratum
d$crude                # the crude estimate, for comparison
d$notes                # the assumption notes printed under the result
d$conf_level
d$method

### A worked use: generating an answer key

Everything a grader needs, in three lines:

In [ ]:
key <- steps_table(mh_odds_ratio(whickham))
answer_key <- setNames(round(key$result, 4), key$symbol)
answer_key[!is.na(answer_key)]

And a worksheet with the answers stripped out, ready to hand to students:

In [ ]:
worksheet <- key[, c("step", "symbol", "label", "formula")]
worksheet$your_answer <- ""
worksheet

---
## Part 15 — Extending the package

`derivation()` and `derivation_step()` are exported, which means **you can add
a measure the package does not have** and it will print, tabulate, and check
exactly like a built-in one.

This is how you should handle a method you teach that is not covered — write it
once, and it behaves like the rest.

### A worked example: positive predictive value

Screening metrics are not in `epibyhand`. Let's add one.

In [ ]:
positive_predictive_value <- function(x, ...) {
  x <- epi2x2(x, ...)

  TP <- x$a; FP <- x$b; FN <- x$c; TN <- x$d
  N    <- TP + FP + FN + TN
  sens <- TP / (TP + FN)
  spec <- TN / (FP + TN)
  prev <- (TP + FN) / N
  ppv  <- TP / (TP + FP)

  derivation(
    method   = "Positive predictive value",
    estimate = ppv,
    symbol   = "PPV",
    data     = x,
    steps = list(
      derivation_step(
        label = "Sensitivity", symbol = "Sens",
        formula     = "TP / (TP + FN)",
        substituted = paste0(TP, " / (", TP, " + ", FN, ")"),
        result      = sens),
      derivation_step(
        label = "Specificity", symbol = "Spec",
        formula     = "TN / (FP + TN)",
        substituted = paste0(TN, " / (", FP, " + ", TN, ")"),
        result      = spec),
      derivation_step(
        label = "Prevalence", symbol = "Prev",
        formula     = "(TP + FN) / N",
        substituted = paste0("(", TP, " + ", FN, ") / ", N),
        result      = prev),
      derivation_step(
        label = "Positive predictive value", symbol = "PPV",
        formula     = "TP / (TP + FP)",
        substituted = paste0(TP, " / (", TP, " + ", FP, ")"),
        result      = ppv,
        note = paste("By Bayes' theorem this equals Sens*Prev /",
                     "(Sens*Prev + (1-Spec)*(1-Prev)). PPV depends on",
                     "prevalence; sensitivity and specificity do not."))
    ),
    notes = paste("Sensitivity and specificity are properties of the test.",
                  "PPV is a property of the test AND the population it is",
                  "used in.")
  )
}

A screening test with 90% sensitivity and 90% specificity, applied to a
population where 5% actually have the disease:

In [ ]:
options(epibyhand.verbose = 2)

screening <- epi2x2(90, 180, 10, 1720,
                    exposure = c("Test positive", "Test negative"),
                    outcome  = c("Diseased", "Healthy"))

positive_predictive_value(screening)

**PPV = 0.33.** A test that is 90% sensitive and 90% specific, applied to a
population with 5% prevalence, is **wrong two times out of three when it says
you are sick**. This is the single most counter-intuitive result in
introductory epidemiology, and seeing the prevalence sitting there in step 3 is
what makes it land.

Your function is now a first-class citizen. Everything works on it:

In [ ]:
p <- positive_predictive_value(screening)

estimate(p)
steps_table(p)[, c("symbol", "result")]
check_work(p, 0.90, step = "Sens")
print(p, verbose = 1)

### The pattern

1. Compute your numbers.
2. Wrap each intermediate in `derivation_step(label, formula, substituted,
   result, symbol, note)`.
3. Wrap the list in `derivation(method, estimate, symbol, data, steps, notes)`.

That is the whole API. `derivation_step()` also takes a `table` argument for a
per-unit grid, which is how the Mantel-Haenszel weights are displayed.

Because all display logic lives in one print method, you write arithmetic and
never write display code.

---
## Part 16 — A complete analysis, start to finish

The whole workflow on one dataset, in the order you would actually do it.

In [ ]:
options(epibyhand.verbose = 0)

# --- 1. Look at the crude association --------------------------------
cat("CRUDE\n")
cat("  OR:", estimate(odds_ratio(whickham_crude)),
    " CI:", confint(odds_ratio(whickham_crude)), "\n")
cat("  RR:", estimate(risk_ratio(whickham_crude)),
    " CI:", confint(risk_ratio(whickham_crude)), "\n")

# --- 2. Inspect strata BEFORE pooling ---------------------------------
cat("\nSTRATUM-SPECIFIC ODDS RATIOS\n")
print(round(mh_odds_ratio(whickham)$stratum_estimates, 3))

# --- 3. Test the pooling assumption -----------------------------------
h <- homogeneity(whickham)
cat("\nHOMOGENEITY (Breslow-Day, Tarone corrected)\n")
cat("  X2 =", round(estimate(h), 3),
    " p =", round(steps_table(h)$result[4], 3), "\n")

# --- 4. Pool -----------------------------------------------------------
mh <- mh_odds_ratio(whickham)
cat("\nADJUSTED\n")
cat("  OR_MH:", estimate(mh), " CI:", confint(mh), "\n")

# --- 5. Quantify the confounding ---------------------------------------
cat("\nCONFOUNDING\n")
cat("  crude    :", mh$crude, "\n")
cat("  adjusted :", estimate(mh), "\n")
cat("  change   :", round(100 * (mh$crude - estimate(mh)) / estimate(mh), 1),
    "%\n")

Five steps, in the only order that is defensible:

1. **Crude first**, so you can see what you would have concluded without
   thinking.
2. **Strata before pooling** — always. A pooled number computed before you look
   at what it is pooling is a number you cannot defend.
3. **Homogeneity**, to check that pooling means anything at all.
4. **Pool**, if step 3 allowed it.
5. **Quantify the confounding**, so the reader can see what adjustment did
   rather than taking your word for it.

---
## Function reference

| Function | Purpose |
|---|---|
| `epi2x2(a, b, c, d)` | Build a 2 x 2 table from counts or a matrix |
| `epi_strata(...)` | Build stratified tables from counts, a list, or an array |
| `collapse_strata(x)` | Add strata cell by cell to recover the crude table |
| `risk_ratio(x)` | Risk ratio, log-scale interval |
| `risk_difference(x)` | Risk difference, number needed to expose |
| `odds_ratio(x)` | Odds ratio, Woolf interval |
| `attributable_fraction(x, among)` | AFe or PAF, three formulas shown to agree |
| `mh_odds_ratio(x)` | Mantel-Haenszel OR, Robins-Breslow-Greenland interval |
| `mh_risk_ratio(x)` | Mantel-Haenszel RR, Greenland-Robins interval |
| `homogeneity(x, tarone)` | Breslow-Day test, Tarone corrected by default |
| `check_work(x, value, step, tol)` | Locate where a hand calculation diverged |
| `steps_table(x)` | Derivation as a data frame |
| `estimate(x)` | Extract the point estimate |
| `confint(x)` | Extract the interval (ignores `level` — see Part 5) |
| `derivation()`, `derivation_step()` | Build your own measure |

### Options

| Option | Effect |
|---|---|
| `epibyhand.verbose` | `0` result only, `1` add formulas, `2` full working (default) |
| `epibyhand.digits` | Significant digits, default `4` |

Both can be overridden per call: `print(d, verbose = 1, digits = 2)`.

### Arguments common to the measures

| Argument | Where | Effect |
|---|---|---|
| `conf_level` | all measures | Confidence level, default `0.95` |
| `exposure`, `outcome` | `epi2x2()`, `epi_strata()` | Row and column labels |
| `labels` | `epi_strata()` | Stratum names |
| `among` | `attributable_fraction()` | `"exposed"` (default) or `"population"` |
| `tarone` | `homogeneity()` | Apply Tarone's correction, default `TRUE` |
| `step`, `tol` | `check_work()` | Target step and relative tolerance |

---
---
# Exercises

Work each one before opening the answer. The explanations are where most of the
learning is.

## Exercise 1 — Build a table and check your arithmetic

A case-control study of a rare cancer recruits **90 cases** and **120
controls**. Among the cases, **60** report the exposure. Among the controls,
**40** do.

1. Build the 2 x 2 table with `epi2x2()`.
2. Compute the odds ratio *by hand* before running anything.
3. Verify with `check_work()`.
4. Should you also compute a risk ratio? Why or why not?

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
cc <- epi2x2(60, 40, 30, 80,
             exposure = c("Exposed", "Unexposed"),
             outcome  = c("Case", "Control"))

d <- odds_ratio(cc)
check_work(d, 4.0)
```

**The table.** 90 cases of whom 60 exposed leaves 30 unexposed cases. 120
controls of whom 40 exposed leaves 80 unexposed controls. In a, b, c, d order
that is **60, 40, 30, 80**.

**By hand.** `OR = ad/bc = (60 x 80)/(40 x 30) = 4800/1200 = 4.0`. Equivalently
`odds1/odds0 = (60/40)/(30/80) = 1.5/0.375 = 4.0`.

**No risk ratio.** This is the important part. The package will happily compute
one — it gets 2.2 — but that number is **meaningless**. The investigators chose
to recruit 90 cases and 120 controls; that ratio is an artifact of study design,
not of nature. Recruiting 240 controls instead would change the "risk ratio"
and leave the odds ratio untouched.

This is exactly why `epi2x2()` lets you label the columns `Case`/`Control`. The
labels are the reminder. Note too that `odds_ratio()` will *not* print its
"these data are tabulated as a cohort" note here — it only offers the risk
ratio comparison when the table plausibly supports one.

</details>

## Exercise 2 — When is an odds ratio a risk ratio?

A trial reports an odds ratio of **2.5** for a side effect.

1. If the side effect occurs in **1%** of the control group, roughly what is
   the risk ratio?
2. If it occurs in **40%** of the control group, roughly what is the risk
   ratio?
3. A journalist writes "people on the drug were 2.5 times as likely to have the
   side effect." When is that fair?

*Hint: `R1 = (OR x R0) / (1 - R0 + OR x R0)`, then `RR = R1/R0`.*

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
or_to_rr <- function(OR, R0) {
  R1 <- (OR * R0) / (1 - R0 + OR * R0)
  R1 / R0
}

or_to_rr(2.5, 0.01)   # 2.463
or_to_rr(2.5, 0.40)   # 1.786
```

**1. R0 = 1%** → RR ≈ **2.46**. Essentially the same as the odds ratio.

**2. R0 = 40%** → RR ≈ **1.79**. The odds ratio overstates the risk ratio by
about 40%.

**3.** The journalist is fair in case 1 and **wrong in case 2**. "2.5 times as
likely" is a statement about risk, and when the outcome is common the odds
ratio is simply not that number.

The general rule: the odds ratio is always further from 1 than the risk ratio,
and they converge only when the outcome is rare in *both* groups. The
convenient threshold is around 10%, but there is nothing magic about it — the
divergence is continuous, as the gradient in Part 4 showed.

This is not a pedantic point. Misreporting odds ratios as risk ratios is one of
the most common statistical errors in published health journalism, and it
systematically exaggerates effect sizes.

</details>

## Exercise 3 — Two different questions

Two exposures in the same population of 10,000 people:

* **Exposure A**: risk 0.20 in the exposed, 0.10 in the unexposed. 50% of the
  population is exposed.
* **Exposure B**: risk 0.50 in the exposed, 0.05 in the unexposed. 1% of the
  population is exposed.

1. Which has the larger risk ratio?
2. Which has the larger population attributable fraction?
3. You run a health department with money for exactly one campaign. Which do
   you target?

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
options(epibyhand.verbose = 0)

A <- epi2x2(1000, 4000,  500, 4500)   # 5000 exposed, 5000 unexposed
B <- epi2x2(  50,   50,  495, 9405)   # 100 exposed, 9900 unexposed

estimate(risk_ratio(A))                                    # 2
estimate(risk_ratio(B))                                    # 10
estimate(attributable_fraction(A, among = "population"))   # 0.333
estimate(attributable_fraction(B, among = "population"))   # 0.083
```

**1. Exposure B**, by a wide margin: RR = 10 versus RR = 2.

**2. Exposure A**, also by a wide margin: PAF ≈ 33% versus ≈ 8%.

**3. Exposure A** — assuming the campaigns are equally effective and equally
expensive.

B is a much more *dangerous* exposure, but it touches 1% of the population. A
is only moderately dangerous but half the population has it. Eliminating A
prevents about a third of all cases; eliminating B prevents about one in twelve.

This is the single most useful thing the attributable fraction does. The risk
ratio answers *"is this exposure harmful?"* — a question about biology. The PAF
answers *"how much of our disease burden does it cause here?"* — a question
about this population, whose answer changes when you cross a border.

A caveat worth stating: this assumes both campaigns would be equally successful
at removing the exposure. If A is deeply embedded in the culture and B is one
contaminated water source you could fix on Tuesday, the calculus changes. PAF
tells you the size of the prize, not the cost of winning it.

</details>

## Exercise 4 — Diagnose the discrepancy

A colleague sends you a stratified analysis and asks what to report.

```
Stratum 1:  a=30, b=70, c=20, d=80
Stratum 2:  a=60, b=40, c=45, d=55
```

1. Compute the crude odds ratio and the Mantel-Haenszel odds ratio.
2. Run the homogeneity test.
3. Is the stratifying variable a confounder, an effect modifier, both, or
   neither? What do you tell your colleague to report?

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
s <- epi_strata(c(30, 70, 20, 80), c(60, 40, 45, 55),
                labels = c("Stratum 1", "Stratum 2"))

d <- mh_odds_ratio(s)
round(d$stratum_estimates, 3)   # 1.714 and 1.833
d$crude                         # 1.891
estimate(d)                     # 1.780
homogeneity(s)                  # X2 = 0.043, p = 0.84
```

**Neither, to any consequential degree.**

* Stratum estimates **1.71** and **1.83** are close together, and Breslow-Day
  gives p = 0.84. **No effect modification.**
* Crude **1.89** versus adjusted **1.78** is a change of about 6%, below the
  usual 10% working threshold. **No meaningful confounding.**

Tell your colleague either number is defensible, and to report the adjusted
one — 1.78 — because adjusting when you did not need to costs almost nothing,
whereas failing to adjust when you did need to is a real error.

**The wider point.** Stratifying and finding nothing is a perfectly good result,
and it is the most common one. The Whickham example is memorable precisely
because reversals are *rare*. A student who has only ever seen Simpson's
paradox examples will over-read every small difference between crude and
adjusted estimates as meaningful.

Note also that the 10% rule is a convention, not a law. It is a judgement about
whether the shift matters for your substantive question — which is why it is
not, and must not be, a hypothesis test.

</details>

## Exercise 5 — Find the student's error

Three students submit answers for the odds ratio of `epi2x2(36, 14, 30, 25)`.
The correct answer is 2.1429.

* Amara writes **0.72**
* Ben writes **2.5714**
* Chidi writes **1.32**

Use `check_work()` to work out what each did, then say what feedback each needs.

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
d <- odds_ratio(epi2x2(36, 14, 30, 25))

check_work(d, 0.72)     # matches nothing in this derivation
check_work(d, 2.5714)   # matches step 1: odds among the exposed
check_work(d, 1.32)     # matches nothing here either
```

**Ben** computed `36/14 = 2.5714`, the odds among the exposed, and stopped.
`check_work()` says so explicitly and names the next step. His feedback is "you
were most of the way there, finish the division" — a one-sentence fix.

**Amara** wrote 0.72, which is `36/50` — the *risk* among the exposed, not the
odds. `check_work()` does not find it, because it is not a step in this
derivation at all. Her error is conceptual: she is dividing by the row total
instead of by the other cell. She needs the definition of odds retaught, not
her arithmetic checked.

**Chidi** wrote 1.32, which is the *risk ratio* for this table. He answered a
different question. His feedback is about reading the question and about when
each measure is appropriate.

**Why this matters.** All three are "wrong" in a gradebook, but they need three
completely different conversations. Ben is 30 seconds from correct; Amara has a
definitional gap; Chidi has a conceptual one. Software that only says "the
answer is 2.1429" flattens all three into the same non-feedback.

Note the limit too: `check_work()` can only find errors that correspond to a
step it actually computed. Amara's and Chidi's errors are invisible to it — the
tool narrows the search, it does not replace a teacher looking at the work.

</details>

## Exercise 6 — Reconstruct a published result

A paper reports: *"Among the 400 exposed workers, 120 developed the condition.
The risk ratio compared with 600 unexposed workers was 2.0."*

1. Reconstruct the full 2 x 2 table.
2. Compute the attributable fraction among the exposed.
3. The paper claims eliminating the exposure would prevent 50% of all cases in
   this workforce. Check it.

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
# R1 = 120/400 = 0.30.  RR = 2 so R0 = 0.15, giving 90 cases among 600.
tab <- epi2x2(120, 280, 90, 510)

options(epibyhand.verbose = 0)
estimate(attributable_fraction(tab, among = "exposed"))      # 0.5
estimate(attributable_fraction(tab, among = "population"))   # 0.2857
```

**1. The table.** `R1 = 120/400 = 0.30`. Since RR = 2, `R0 = 0.15`, so
`0.15 x 600 = 90` cases among the unexposed. Cells: **120, 280, 90, 510**.

**2. AFe = 0.50.** Half the illness among exposed workers is attributable to
the exposure. Sanity check: `(RR - 1)/RR = 1/2`. Whenever the risk ratio is
exactly 2, the AFe is exactly 50%.

**3. The claim is wrong.** It quotes the AFe but describes the PAF.

The PAF is **0.286**. Eliminating the exposure would prevent about 29% of cases
in this workforce, not 50%, because 60% of the workforce is unexposed and
contributes cases that have nothing to do with the exposure.

This is a real and frequent error in occupational and environmental health
reporting, and it always errs in the same direction — overstating the benefit
of an intervention. The tell is the phrase **"of all cases"**: that is a
population claim, so it needs a population fraction.

</details>

## Exercise 7 — Choose the measure

For each scenario, say which measure you would report and why.

1. A case-control study of a rare congenital defect and a medication taken in
   pregnancy.
2. A vaccine trial where 2% of the placebo arm and 0.5% of the vaccine arm are
   infected.
3. A city deciding whether to fund a smoking cessation programme.
4. A cohort study where 60% of the exposed and 30% of the unexposed develop
   hypertension.

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

**1. Odds ratio.** In a case-control design it is the only valid measure of
association — risk is not estimable when you sample on the outcome. And because
the defect is rare, the OR approximates the risk ratio, so it can be
*interpreted* as one.

**2. Risk ratio**, or its complement, vaccine efficacy. `RR = 0.005/0.02 =
0.25`, so efficacy is `1 - RR = 75%`. This is a trial, so risks are estimable
and there is no reason to reach for odds. Report the risk difference alongside
it — 1.5 percentage points — because that is what determines how many people
must be vaccinated to prevent one infection.

**3. Population attributable fraction**, plus the risk difference. The city is
not asking "is smoking harmful?" — that is settled. It is asking "how much of
our disease burden would this programme remove?", which is a question about
impact in this population.

**4. Risk ratio and risk difference — not the odds ratio.** With 60% and 30%
outcomes the OR would be 3.5 while the RR is 2.0, and almost every reader will
misinterpret the 3.5. When risks are estimable and the outcome is common, the
odds ratio is the wrong choice for communication even though it is not wrong
arithmetically.

**The general principle.** The measure follows from two things: the study
design, which determines what is *estimable*, and the question, which determines
what is *relevant*. Ratios speak to causation; differences and attributable
fractions speak to impact.

</details>

## Exercise 8 — Design your own confounding

Construct a 2 x 2 x 2 stratified dataset from scratch in which:

* the crude odds ratio is **below 1**,
* both stratum-specific odds ratios are **above 1**,
* the Breslow-Day test does **not** reject homogeneity.

Verify with the package. Then explain what makes it work.

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

One construction:

```r
mine <- epi_strata(
  c( 20, 180,   5,  95),   # low-risk stratum,  mostly unexposed
  c(160,  40,  75,  25),   # high-risk stratum, mostly exposed
  labels = c("Low risk", "High risk")
)

d <- mh_odds_ratio(mine)
round(d$stratum_estimates, 3)   # 2.111 and 1.333
d$crude                         # 0.847
estimate(d)                     # 1.512
homogeneity(mine)               # p comfortably above 0.05
```

**What makes it work** is two conditions holding at once — which is precisely
the definition of a confounder:

1. **The stratifying variable is associated with the outcome.** Baseline risk is
   about 5% in the low-risk stratum and 75% in the high-risk one.
2. **The stratifying variable is associated with the exposure.** The low-risk
   stratum is 67% unexposed; the high-risk stratum is 67% exposed.

Collapsing pools the mostly-unexposed low-risk people with the mostly-exposed
high-risk people. The unexposed group ends up dominated by low-risk
individuals, so it looks artificially healthy, and the exposure looks protective
by comparison.

**Worth noticing:** this took deliberate effort to construct. Full reversals
need both associations to be strong and to point in cooperating directions. That
is why Whickham is famous. If you had to work this hard to build one, be
correspondingly sceptical when you think you have found one in real data — check
your table orientation first.

</details>

## Exercise 9 — A zero cell

A small outbreak investigation gives this table:

|  | Ill | Well |
|---|---|---|
| **Ate the shellfish** | 12 | 0 |
| **Did not** | 3 | 15 |

1. Compute the odds ratio. What happens?
2. Compute the risk ratio. Does the same thing happen?
3. Apply a continuity correction and report both answers. Which would you put
   in a paper?

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
options(epibyhand.verbose = 0)

tab <- epi2x2(12, 0, 3, 15,
              exposure = c("Ate shellfish", "Did not"),
              outcome  = c("Ill", "Well"))

estimate(odds_ratio(tab))    # Inf
confint(odds_ratio(tab))     # NaN, Inf
estimate(risk_ratio(tab))    # 6
confint(risk_ratio(tab))     # finite!

corrected <- epi2x2(12.5, 0.5, 3.5, 15.5)
estimate(odds_ratio(corrected))   # about 110.7
```

**1. The odds ratio is `Inf`** with a `NaN` lower limit. Cell b is zero, so
`ad/bc` divides by zero. The package says so and explains what a continuity
correction would do rather than applying one silently.

**2. The risk ratio is fine — 6.0, with a usable interval.** This surprises
people. The risk ratio divides by *row totals*, which are non-zero, so a single
empty cell does not destroy it. Only the odds ratio needs every individual cell.

**3.** With the Haldane-Anscombe correction the odds ratio becomes about **111**
— an enormous and largely arbitrary number, since it is driven by the 0.5 you
invented rather than by data.

**Report the risk ratio.** These are cohort data, risks are estimable, the risk
ratio needs no correction, and it answers the question you actually care about.
Reaching for a corrected odds ratio here would be choosing the measure that
requires a fudge over the one that does not.

If you must report a corrected odds ratio — for meta-analysis, say — state the
correction in your methods. It biases toward the null and its interval is
approximate. The general lesson: **a zero cell is information about your study,
usually that it is small. It is not a technical nuisance to be corrected away.**

</details>

## Exercise 10 — Add a measure

The package has no function for the **prevented fraction**, the mirror image of
the attributable fraction used when an exposure is protective:

```
PFe = 1 - RR        (among the exposed)
```

Write `prevented_fraction()` using `derivation()` and `derivation_step()`, so
that it prints a full derivation, works with `steps_table()`, and works with
`check_work()`.

Test it on a vaccine trial: 25 infections among 500 vaccinated, 100 among 500
unvaccinated.

In [ ]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
prevented_fraction <- function(x, ...) {
  x <- epi2x2(x, ...)

  R1 <- x$a / (x$a + x$b)
  R0 <- x$c / (x$c + x$d)
  RR <- R1 / R0
  PF <- 1 - RR

  derivation(
    method   = "Prevented fraction among the exposed",
    estimate = PF,
    symbol   = "PFe",
    data     = x,
    steps = list(
      derivation_step(
        label = "Risk among the exposed", symbol = "R1",
        formula     = "a / (a + b)",
        substituted = paste0(x$a, " / (", x$a, " + ", x$b, ")"),
        result      = R1),
      derivation_step(
        label = "Risk among the unexposed", symbol = "R0",
        formula     = "c / (c + d)",
        substituted = paste0(x$c, " / (", x$c, " + ", x$d, ")"),
        result      = R0),
      derivation_step(
        label = "Risk ratio", symbol = "RR",
        formula     = "R1 / R0",
        substituted = paste0(round(R1, 4), " / ", round(R0, 4)),
        result      = RR),
      derivation_step(
        label = "Prevented fraction", symbol = "PFe",
        formula     = "1 - RR",
        substituted = paste0("1 - ", round(RR, 4)),
        result      = PF,
        note = paste("Only meaningful when RR < 1. If RR > 1 the exposure is",
                     "harmful and you want the attributable fraction instead."))
    ),
    notes = paste("In a vaccine trial this quantity is vaccine efficacy.")
  )
}

trial <- epi2x2(25, 475, 100, 400,
                exposure = c("Vaccinated", "Unvaccinated"),
                outcome  = c("Infected", "Not infected"))

prevented_fraction(trial)
steps_table(prevented_fraction(trial))[, c("symbol", "result")]
check_work(prevented_fraction(trial), 0.05, step = "R1")
```

**The answer.** `R1 = 25/500 = 0.05`, `R0 = 100/500 = 0.20`, `RR = 0.25`, so
`PFe = 0.75` — **75% vaccine efficacy**.

**What the exercise is really teaching.** Three things worth noticing:

*Vaccine efficacy is not a new concept.* It is the prevented fraction with a
different name, which is itself just `1 - RR`. Half of what looks like a
proliferation of measures in epidemiology is one quantity under several names.

*The `note` field is doing real work.* `PFe` is meaningless when `RR > 1`, and
saying so inside the derivation is better than saying it in documentation
nobody reads.

*You did not write any display code.* Because all display logic lives in one
print method, adding a measure means writing arithmetic and steps — the
printing, the verbosity levels, `steps_table()`, and `check_work()` all come
free. That is the architectural payoff of the single-class design, and it is
why the package can grow without its print layer changing.

</details>

---
---
## Where to go next

* **Package documentation** — `help(package = "epibyhand")`
* **The vignette** — `vignette("epibyhand")`, which works the Whickham data
  through as a single continuous narrative
* **CRAN** — https://cran.r-project.org/package=epibyhand
* **Source and issues** — https://github.com/rajsubediresearch/epibyhand

### Reference

The Whickham data are from Appleton, D. R., French, J. M. and Vanderpump,
M. P. J. (1996). Ignoring a covariate: an example of Simpson's paradox.
*The American Statistician* **50**(4), 340-341.

### Citing

```r
citation("epibyhand")
```